In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
dataset = pd.get_dummies(dataset, columns=['sex', 'smoker', 'region'], dtype='float32')

train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)

train_labels = train_dataset.pop('expenses')
test_labels = test_dataset.pop('expenses')

In [ ]:
normalizer = layers.Normalization()
normalizer.adapt(train_dataset.values.astype('float32'))

model = keras.Sequential([
    layers.Input(shape=(train_dataset.shape[1],)),
    normalizer,
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
    layers.Rescaling(scale=float(train_labels.std()), offset=float(train_labels.mean()))
])

model.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='mae',
              metrics=['mae', 'mse'])

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss',
                                           patience=50,
                                           restore_best_weights=True)

history = model.fit(train_dataset,
                    train_labels,
                    epochs=500,
                    validation_split=0.2,
                    callbacks=[early_stop],
                    verbose=0)

In [ ]:
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)